In [10]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [12]:
# Generate evaluation dataset via Claude using few-shot/prefill prompting
# Uses stop sequences to parse clean JSON directly from the model response
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
# Run dataset generation and persist results to dataset.json for reuse
dataset = generate_dataset()
with open("dataset.json","w") as f:
    json.dump(dataset, f, indent=2)
    
dataset

[{'task': "Write a Python function that parses an AWS S3 object key and extracts the bucket name, folder path, and file name from a full S3 URI (e.g., 's3://my-bucket/path/to/file.txt')"},
 {'task': 'Create a JSON CloudFormation template snippet that defines an AWS Lambda function resource with basic configuration including runtime, handler, and an IAM execution role'},
 {'task': "Write a regular expression that matches valid AWS IAM policy ARNs in the format 'arn:aws:iam::ACCOUNT-ID:policy/POLICY-NAME' and captures the account ID and policy name separately"}]

In [14]:
# Format a task into a prompt and get Claude's response
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""Please solve the following task: {test_case["task"]}"""
    messages=[]
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [15]:
# Run a single test case and return the output alongside a score (grading not yet implemented)
def run_test_case(test_case):
    """Call run prompt then the result"""
    output = run_prompt(test_case)
    #TODO - Grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [16]:
# Run the full evaluation loop over every task in the dataset and collect results
def run_eval(test_case):
    """Merges the prompt and test case input, then returns the result"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [17]:
# Load the saved dataset from disk and run the full evaluation pipeline
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

In [18]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a comprehensive solution with multiple approaches and test cases:\n\n```python\nfrom typing import NamedTuple, Dict\nfrom urllib.parse import urlparse\nimport re\n\n\nclass S3ObjectInfo(NamedTuple):\n    \"\"\"Data class to hold S3 object information\"\"\"\n    bucket: str\n    folder_path: str\n    file_name: str\n    full_key: str\n\n\ndef parse_s3_uri_simple(s3_uri: str) -> S3ObjectInfo:\n    \"\"\"\n    Simple approach using string manipulation.\n    \n    Args:\n        s3_uri: Full S3 URI (e.g., 's3://my-bucket/path/to/file.txt')\n    \n    Returns:\n        S3ObjectInfo containing bucket, folder_path, file_name, and full_key\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    # Validate format\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split bucket fro